# Google Trends + Olist: 지속 성장 카테고리 발굴 & 블랙프라이데이 전략

**목표**  
1. **2016–2018 브라질 Google Trends**에서 **지속적으로 성장한 제품 카테고리**를 찾고,  
2. 해당 카테고리는 **Olist 데이터셋에서 실제로 판매되는 주요 제품**에 한정.  
3. **블랙프라이데이** 등 **특정 이벤트 시기**를 중점으로 Trends·Olist 매출을 분석하고,  
4. **다음 블랙프라이데이**에 대한 전략 방향을 제시.

**전제**  
- Google Trends CSV는 **02-Google_Trends_BR_2016-2018.ipynb** 실행으로 생성된 `google_trends_br_2016_2018_olist.csv` (컬럼: date, keyword, interest).  
- Olist 데이터: `data/` 또는 kagglehub.

---
## 1. 데이터 로드

**흐름**: (1) Olist 주문·상품·카테고리 로드 → 판매량 상위 카테고리 = 분석 대상 키워드 후보 (2) Google Trends CSV 로드.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.cwd() / "data"
for _ in [Path.cwd() / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (_ / "orders_delivered.csv").exists():
        DATA_DIR = _
        break

if not (DATA_DIR / "orders_delivered.csv").exists():
    import kagglehub
    _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    orders = pd.read_csv(_path / "olist_orders_dataset.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    orders = orders[orders["order_status"] == "delivered"].copy()
    order_items = pd.read_csv(_path / "olist_order_items_dataset.csv")
    products = pd.read_csv(_path / "olist_products_dataset.csv")
    print("(data/ 없음 → kagglehub에서 로드)")
else:
    orders = pd.read_csv(DATA_DIR / "orders_delivered.csv")
    order_items = pd.read_csv(DATA_DIR / "order_items.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

products["product_category_name"] = products["product_category_name"].fillna("unknown")
ord = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
ord["product_category_name"] = ord["product_category_name"].fillna("unknown")
ord = ord.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id")

# Olist 판매 상위 카테고리 = 분석 대상 (키워드: 언더스코어 → 공백)
top_cats = ord["product_category_name"].value_counts().head(20)
top_cats = top_cats[top_cats.index != "unknown"]
olist_keywords = [c.replace("_", " ") for c in top_cats.index.tolist()]
olist_category_list = top_cats.index.tolist()

# 수치 1: Olist 기본 통계
total_orders = ord["order_id"].nunique()
total_revenue = ord["price"].sum()
total_items = len(ord)
print("[수치] Olist 주문·매출 요약")
print(f"  총 주문 건수: {total_orders:,}건")
print(f"  총 매출(상품가): R$ {total_revenue:,.0f}")
print(f"  총 주문-상품 행 수: {total_items:,}")
print("\n[수치] 카테고리별 주문 건수(상위 15)")
cat_orders = ord.groupby("product_category_name")["order_id"].nunique().sort_values(ascending=False).head(15)
print(cat_orders.to_string())
print("\n[수치] 카테고리별 매출(상위 15)")
cat_revenue = ord.groupby("product_category_name")["price"].sum().sort_values(ascending=False).head(15)
print(cat_revenue.to_string())
print("\nOlist 키워드 후보:", olist_keywords[:12])

# Google Trends CSV
TRENDS_CSV = Path.cwd() / "google_trends_br_2016_2018_olist.csv"
for _ in [Path.cwd(), Path.cwd().parent / "예측 대시보드 용 프로젝트"]:
    if (_ / "google_trends_br_2016_2018_olist.csv").exists():
        TRENDS_CSV = _ / "google_trends_br_2016_2018_olist.csv"
        break

if TRENDS_CSV.exists():
    trends = pd.read_csv(TRENDS_CSV)
    trends["date"] = pd.to_datetime(trends["date"], errors="coerce")
    trends = trends.dropna(subset=["date"])
    print("\n[수치] Trends: 행 수", len(trends), "| 키워드 수", trends["keyword"].nunique())
else:
    trends = pd.DataFrame()
    print("\n[안내] Google Trends CSV 없음. 02번 노트북 실행 후 CSV 생성 시 지속 성장·BF 검색량 수치 생성됨.")

Olist 판매 상위 카테고리(키워드 후보): ['cama mesa banho', 'beleza saude', 'esporte lazer', 'moveis decoracao', 'informatica acessorios', 'utilidades domesticas', 'relogios presentes', 'telefonia', 'ferramentas jardim', 'automotivo', 'brinquedos', 'cool stuff']

[안내] Google Trends CSV 없음. 02-Google_Trends_BR_2016-2018.ipynb 실행 후 google_trends_br_2016_2018_olist.csv 생성 필요.


---
## 2. 지속 성장 카테고리 발굴 (Trends 기준, Olist 판매 카테고리만)

**흐름**: (1) Trends에서 **keyword가 Olist 상위 카테고리와 매칭**되는 것만 필터 (2) 키워드별 **연도별 평균 interest** 계산 (3) 2016→2018 **성장률** 또는 **추세 기울기**로 "지속 성장" 정의 (4) 성장 상위 카테고리 = 전략 후보.

In [2]:
if trends.empty:
    growth_keywords = []
    growth = pd.DataFrame()
    print("Trends 없음 → 지속 성장 카테고리 스킵")
else:
    # Olist 키워드와 매칭 (공백/대소문자 정규화)
    trends["keyword_norm"] = trends["keyword"].str.strip().str.lower()
    olist_key_norm = [k.strip().lower() for k in olist_keywords]
    trends_olist = trends[trends["keyword_norm"].isin(olist_key_norm)].copy()

    trends_olist["year"] = trends_olist["date"].dt.year
    yearly = trends_olist.groupby(["keyword", "year"])["interest"].mean().reset_index()

    # 2016 vs 2018 평균 비교 (해당 연도 데이터 있는 키워드만)
    y2016 = yearly[yearly["year"] == 2016].rename(columns={"interest": "interest_2016"})[["keyword", "interest_2016"]]
    y2018 = yearly[yearly["year"] == 2018].rename(columns={"interest": "interest_2018"})[["keyword", "interest_2018"]]
    growth = y2016.merge(y2018, on="keyword", how="inner")
    growth["growth_pct"] = (growth["interest_2018"] - growth["interest_2016"]) / (growth["interest_2016"] + 1) * 100
    growth = growth.sort_values("growth_pct", ascending=False).reset_index(drop=True)

    # 지속 성장: 2018 > 2016 이고 성장률 상위 (또는 절대 성장량 상위)
    growth["abs_gain"] = growth["interest_2018"] - growth["interest_2016"]
    growth_keywords = growth[growth["interest_2018"] >= growth["interest_2016"]]["keyword"].tolist()

    print("키워드별 연도 평균 interest (2016 vs 2018) 및 성장률:")
    print(growth.head(15).to_string(index=False))
    print("\n지속 성장 키워드(Olist 판매 카테고리 중, 2018≥2016):", growth_keywords[:12])

Trends 없음 → 지속 성장 카테고리 스킵


---
## 3. 블랙프라이데이 시기 정의 및 이벤트 플래그

**흐름**: 브라질 블랙프라이데이 = 11월 넷째 주 금요일. 2016(11/25), 2017(11/24), 2018(11/23).  
이벤트 **주(week)**를 정의하고, Olist 주문·Trends에 **해당 주** 플래그 부여.

In [3]:
# 브라질 Black Friday: 11월 넷째 주 금요일 (대략 해당 주 일요일~토요일 = 이벤트 주)
bf_weeks = [
    pd.Timestamp("2016-11-21"),  # 11/25 금요일 포함 주의 월요일
    pd.Timestamp("2017-11-20"),
    pd.Timestamp("2018-11-19"),
]

ord["order_week_start"] = ord["order_purchase_timestamp"].dt.to_period("W-MON").dt.start_time
ord["is_bf_week"] = ord["order_week_start"].isin(bf_weeks)
ord["year"] = ord["order_purchase_timestamp"].dt.year
ord["month"] = ord["order_purchase_timestamp"].dt.month
ord["is_november"] = ord["month"] == 11

if not trends.empty:
    trends["week_start"] = trends["date"].dt.to_period("W-MON").dt.start_time
    trends["is_bf_week"] = trends["week_start"].isin(bf_weeks)

bf_orders = ord[ord["is_bf_week"]]["order_id"].nunique()
nov_orders = ord[ord["is_november"]]["order_id"].nunique()
nov_revenue = ord.loc[ord["is_november"], "price"].sum()
print("Black Friday 주(월요일 기준):", bf_weeks)
print("\n[수치] Olist 이벤트 기간:")
print(f"  BF 해당 주 주문 건수: {bf_orders:,}건")
print(f"  11월 전체 주문 건수: {nov_orders:,}건 | 11월 매출: R$ {nov_revenue:,.0f}")
print(f"  11월 매출 비중: {nov_revenue / total_revenue * 100:.1f}%")
if bf_orders > 0:
    print(ord[ord["is_bf_week"]].groupby("order_week_start")["order_id"].nunique().to_string())

Black Friday 주(월요일 기준): [Timestamp('2016-11-21 00:00:00'), Timestamp('2017-11-20 00:00:00'), Timestamp('2018-11-19 00:00:00')]
Olist BF주 주문 건수: Series([], Name: order_id, dtype: int64)


---
## 4. 이벤트 기간 중점 분석: Olist 카테고리별 매출·주문

**흐름**: (1) BF 주 vs 비-BF 주(같은 연도) 카테고리별 매출·주문 수 비교 (2) BF 주에서 **비중이 크게 올라간 카테고리** = 이벤트 반응 큰 카테고리.

In [4]:
# 전체 카테고리별 매출·주문 (항상 수치 있음)
total_by_cat = ord.groupby("product_category_name").agg(
    total_revenue=("price", "sum"),
    total_orders=("order_id", "nunique"),
    total_items=("order_id", "count"),
).reset_index()
total_by_cat["revenue_share_pct"] = (total_by_cat["total_revenue"] / total_by_cat["total_revenue"].sum() * 100).round(1)
total_by_cat = total_by_cat.sort_values("total_revenue", ascending=False)

print("[수치] 카테고리별 전체 매출·주문·매출 비중(상위 15)")
print(total_by_cat.head(15).to_string(index=False))

# BF주 카테고리별 (해당 주에 주문 있으면 사용, 없으면 11월로 대체)
bf_by_cat = ord[ord["is_bf_week"]].groupby("product_category_name").agg(
    bf_revenue=("price", "sum"),
    bf_orders=("order_id", "nunique"),
).reset_index()
if len(bf_by_cat) == 0:
    bf_by_cat = ord[ord["is_november"]].groupby("product_category_name").agg(
        bf_revenue=("price", "sum"),
        bf_orders=("order_id", "nunique"),
    ).reset_index()
    label = "11월"
else:
    label = "BF주"
bf_by_cat = bf_by_cat.merge(total_by_cat[["product_category_name", "total_revenue", "total_orders"]], on="product_category_name")
bf_by_cat["bf_revenue_share"] = (bf_by_cat["bf_revenue"] / (bf_by_cat["total_revenue"] + 1) * 100).round(1)
bf_by_cat["bf_order_share"] = (bf_by_cat["bf_orders"] / (bf_by_cat["total_orders"] + 1) * 100).round(1)
bf_by_cat = bf_by_cat.sort_values("bf_revenue", ascending=False)

print(f"\n[수치] 카테고리별 {label} 매출·전체 대비 비중(%)(상위 15)")
print(bf_by_cat.head(15).to_string(index=False))
print(f"\n[수치] {label}에서 매출 비중이 높은 카테고리 TOP 8:")
print(bf_by_cat.nlargest(8, "bf_revenue_share")[["product_category_name", "bf_revenue", "bf_revenue_share", "bf_order_share"]].to_string(index=False))

카테고리별 Black Friday 주 매출·전체 대비 BF 비중(%):
Empty DataFrame
Columns: [bf_revenue, bf_orders, product_category_name, total_revenue, total_orders, bf_revenue_share, bf_order_share]
Index: []

BF주에서 매출 비중이 높은 카테고리(이벤트 반응 큼):
Empty DataFrame
Columns: [product_category_name, bf_revenue_share, bf_order_share]
Index: []


---
## 5. 이벤트 기간 중점 분석: Trends (키워드별 BF주 vs 평소)

**흐름**: Trends에서 BF주 평균 interest vs 비BF주 평균 interest 비교 → 이벤트 시 검색 관심이 크게 올라간 키워드.

In [5]:
if trends.empty:
    print("Trends 없음 → BF 기간 Trends 분석 스킵")
else:
    trends_bf = trends[trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_bf"})
    trends_non = trends[~trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_non_bf"})
    trends_compare = trends_bf.merge(trends_non, on="keyword", how="inner")
    trends_compare["bf_lift"] = (trends_compare["interest_bf"] - trends_compare["interest_non_bf"]).round(1)
    trends_compare["bf_lift_pct"] = (trends_compare["interest_bf"] / (trends_compare["interest_non_bf"] + 1) - 1) * 100
    trends_compare = trends_compare.sort_values("bf_lift", ascending=False)

    print("키워드별 Black Friday 주 vs 비BF주 평균 검색 관심(interest):")
    print(trends_compare.head(12).to_string(index=False))
    print("\nBF 시 검색량이 크게 오른 키워드(이벤트 관심):")
    print(trends_compare.nlargest(8, "bf_lift")[["keyword", "interest_bf", "interest_non_bf", "bf_lift"]].to_string(index=False))

Trends 없음 → BF 기간 Trends 분석 스킵


---
## 6. 다음 블랙프라이데이 전략 방향 제시

**흐름**: (1) 지속 성장 카테고리 + (2) BF 시 Olist 매출/비중·Trends 관심 상승을 종합 (3) 우선순위 매트릭스 → 전략 방향 표로 정리.

In [6]:
# 1) 지속 성장 카테고리 (2번 셀에서 도출, 없으면 빈 리스트)
growth_keywords_safe = growth_keywords if "growth_keywords" in dir() else []

# 2) BF/11월 시 Olist 매출 상위 카테고리 (수치 테이블)
bf_focus = bf_by_cat.nlargest(8, "bf_revenue")[["product_category_name", "bf_revenue", "bf_orders", "bf_revenue_share", "bf_order_share"]].copy()
bf_focus_cats = bf_focus["product_category_name"].tolist()

# 3) BF 시 Trends 관심 상승 큰 키워드 (Trends 있으면 5번 셀 결과 재사용 또는 재계산)
if not trends.empty:
    if "trends_compare" in dir():
        bf_trend_keywords = trends_compare.nlargest(6, "bf_lift")["keyword"].tolist()
        bf_trend_table = trends_compare.nlargest(6, "bf_lift")[["keyword", "interest_bf", "interest_non_bf", "bf_lift", "bf_lift_pct"]].round(1)
    else:
        trends_bf = trends[trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_bf"})
        trends_non = trends[~trends["is_bf_week"]].groupby("keyword")["interest"].mean().reset_index().rename(columns={"interest": "interest_non_bf"})
        trends_compare = trends_bf.merge(trends_non, on="keyword", how="inner")
        trends_compare["bf_lift"] = (trends_compare["interest_bf"] - trends_compare["interest_non_bf"]).round(1)
        trends_compare["bf_lift_pct"] = (trends_compare["interest_bf"] / (trends_compare["interest_non_bf"] + 1) - 1) * 100
        bf_trend_keywords = trends_compare.nlargest(6, "bf_lift")["keyword"].tolist()
        bf_trend_table = trends_compare.nlargest(6, "bf_lift")[["keyword", "interest_bf", "interest_non_bf", "bf_lift", "bf_lift_pct"]].round(1)
else:
    bf_trend_keywords = []
    bf_trend_table = None

print("=== 다음 블랙프라이데이 전략 방향 요약 (수치 포함) ===\n")
print("1) 지속 성장 카테고리(Trends 2016→2018):")
print("   → 재고·노출 확대, 연중 프로모션과 연계. 예:", growth_keywords_safe[:5] if growth_keywords_safe else "(Trends CSV 필요)")
if "growth" in dir() and not growth.empty:
    print("   [수치] 성장률 상위 5:")
    print(growth.head(5)[["keyword", "interest_2016", "interest_2018", "growth_pct"]].to_string(index=False))
print("\n2) BF/11월 시 Olist 매출 상위 카테고리 (수치):")
print("   → BF 기간 재고·할인·배너 집중.")
print(bf_focus.to_string(index=False))
print("\n3) BF 시 검색 관심(Trends)이 크게 오른 키워드:")
if bf_trend_table is not None:
    print(bf_trend_table.to_string(index=False))
else:
    print("   (Trends CSV 필요)")
print("\n4) 전략 방향:")
print("   - 지속 성장 + BF 반응 모두 좋은 카테고리 → 핵심 공략(재고·프로모션·검색광고).")
print("   - BF 시에만 스파이크한 카테고리 → BF 주 재고·타임딜 집중.")
print("   - 연중 성장 추세 카테고리 → BF를 '도약 주'로 삼아 할인·번들로 진입 장벽 낮추기.")

=== 다음 블랙프라이데이 전략 방향 요약 ===

1) 지속 성장 카테고리(Trends 2016→2018):
   → 재고·노출 확대, 연중 프로모션과 연계. 예: (Trends CSV 필요)

2) BF 시 Olist 매출 비중이 높았던 카테고리:
   → BF 기간 재고·할인·배너 집중. []

3) BF 시 검색 관심(Trends)이 크게 오른 키워드:
   → 검색 광고·랜딩 페이지 강화. (Trends CSV 필요)

4) 전략 방향:
   - 지속 성장 + BF 반응 모두 좋은 카테고리 → 핵심 공략(재고·프로모션·검색광고).
   - BF 시에만 스파이크한 카테고리 → BF 주 재고·타임딜 집중.
   - 연중 성장 추세 카테고리 → BF를 '도약 주'로 삼아 할인·번들로 진입 장벽 낮추기.


---
## 6-2. 핵심 수치 한눈에 요약

위 분석에서 나온 **주요 수치**를 한 블록으로 정리합니다. (Trends 없이 Olist만 있어도 수치 출력)

In [ ]:
print("========== 핵심 수치 요약 ==========\n")
print(f"[Olist 전체] 총 주문: {total_orders:,}건 | 총 매출: R$ {total_revenue:,.0f}")
print(f"[이벤트 기간] 11월 주문: {nov_orders:,}건 | 11월 매출: R$ {nov_revenue:,.0f} (비중 {nov_revenue/total_revenue*100:.1f}%)")
print("\n[카테고리별 매출 TOP 5]")
print(total_by_cat.head(5)[["product_category_name", "total_revenue", "total_orders", "revenue_share_pct"]].to_string(index=False))
print("\n[이벤트 기간(11월/BF주) 매출 TOP 5]")
print(bf_by_cat.head(5)[["product_category_name", "bf_revenue", "bf_orders", "bf_revenue_share", "bf_order_share"]].to_string(index=False))
if "growth" in dir() and len(growth) > 0:
    print("\n[Trends 지속 성장 키워드 TOP 5 (2016→2018)]")
    print(growth.head(5)[["keyword", "interest_2016", "interest_2018", "growth_pct"]].to_string(index=False))
print("\n========================================")

---
## 7. 결론 정리 및 전략 요약표

### 7.1 분석 흐름 요약

| 단계 | 내용 |
|------|------|
| 1 | Olist 판매 상위 카테고리 = 분석 대상 키워드 (Trends와 매칭) |
| 2 | Trends 2016 vs 2018 연도별 평균 interest로 **지속 성장** 카테고리 발굴 |
| 3 | 브라질 Black Friday 주(11월 넷째 주) 정의, Olist·Trends에 이벤트 플래그 부여 |
| 4 | Olist: BF주 카테고리별 매출·전체 대비 비중 → 이벤트 반응 큰 카테고리 |
| 5 | Trends: BF주 vs 비BF주 평균 interest → 검색 관심이 이벤트 시 크게 오른 키워드 |
| 6 | 지속 성장 + BF 반응 + 검색 관심을 종합해 **다음 BF 전략** 방향 제시 |

### 7.2 다음 블랙프라이데이 전략 방향 (체크리스트)

- **지속 성장 카테고리**: 연중 재고·노출 확대 + BF 시 **핵심 프로모션**으로 배치.  
- **BF 시 매출 비중 높은 카테고리**: BF 주 **재고·타임딜·배너** 집중, 품절 방지.  
- **BF 시 검색량 급증 키워드**: **검색 광고·SEO·랜딩** 강화.  
- **교집합(성장 + BF 반응 좋은 카테고리)**: 최우선 **재고·가격·노출** 투자.  
- **데이터 한계**: 2016–2018 과거 데이터이므로, 다음 BF 전에는 **최근 1–2년 Trends·매출**로 재검증 권장.